IMPORT & LOAD DATA

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load raw JSON (same source as ranking.py)
review_file = '../data/Luxury_Beauty_5.json'
reviews = []
with open(review_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading reviews"):
        r = json.loads(line.strip())
        reviews.append({
            'reviewerID': r.get('reviewerID'),
            'asin': r.get('asin'),
            'overall': r.get('overall'),
            'unixReviewTime': r.get('unixReviewTime'),
            'reviewText': r.get('reviewText', ''),
            'summary': r.get('summary', '')
        })

df = pd.DataFrame(reviews)
print(f"Raw data: {df.shape}")

# --- Deduplicate (same review crawled multiple times) ---
before_dedup = len(df)
df = df.drop_duplicates(subset=['reviewerID', 'asin', 'unixReviewTime'], keep='first')
df = df.reset_index(drop=True)
print(f"After dedup: {df.shape} (removed {before_dedup - len(df):,} duplicates)")

# --- Filtering (match ranking.py: min_user=5, min_item=3) ---
min_user = 5
min_item = 3

print(f"\nFiltering: min_user={min_user}, min_item={min_item}")
user_counts = df['reviewerID'].value_counts()
valid_users = user_counts[user_counts >= min_user].index
df = df[df['reviewerID'].isin(valid_users)]

item_counts = df['asin'].value_counts()
valid_items = item_counts[item_counts >= min_item].index
df = df[df['asin'].isin(valid_items)]
df = df.reset_index(drop=True)

print(f"After filtering: {df.shape}")
print(f"Users: {df['reviewerID'].nunique()}, Items: {df['asin'].nunique()}")
df.head()

Loading reviews: 34278it [00:00, 143383.07it/s]


Raw data: (34278, 6)
After dedup: (29059, 6) (removed 5,219 duplicates)

Filtering: min_user=5, min_item=3
After filtering: (25206, 6)
Users: 2782, Items: 1399


,reviewerID,asin,overall,unixReviewTime,reviewText,summary
0,A2HOI48JK8838M,B00004U9V2,5.0,1515110400,This handcream has a beautiful fragrance. It d...,Beautiful Fragrance
1,A1YIPEY7HX73S7,B00004U9V2,5.0,1491350400,"wonderful hand lotion, for seriously dry skin,...",wonderful hand lotion
2,A2QCGHIJ2TCLVP,B00004U9V2,5.0,1490572800,"Best hand cream around. Silky, thick, soaks i...",Best hand cream around
3,A2R4UNHFJBA6PY,B00004U9V2,5.0,1489968000,Thanks!!,Five Stars
4,A2QCGHIJ2TCLVP,B00004U9V2,5.0,1488240000,Great hand lotion. Soaks right in and leaves ...,Great hand lotion!


Train–Test Split + Ground Truth

In [3]:
# Sort by (user, time, item) for DETERMINISTIC ordering (match ranking.ipynb)
df = df.sort_values(["reviewerID", "unixReviewTime", "asin"])

train_rows = []
ground_truth = {}

for user, g in df.groupby("reviewerID"):
    items = g["asin"].tolist()

    if len(items) <= 7:
        ground_truth[user] = items[-1:]      # leave-1
        train_rows.append(g.iloc[:-1])
    else:
        ground_truth[user] = items[-2:]      # leave-2
        train_rows.append(g.iloc[:-2])

train_df = pd.concat(train_rows).reset_index(drop=True)

print("Train interactions:", len(train_df))
print("Users:", train_df["reviewerID"].nunique())
print("Ground truth users:", len(ground_truth))
print("Avg test items/user:", np.mean([len(v) for v in ground_truth.values()]))

Train interactions: 21076
Users: 2782
Ground truth users: 2782
Avg test items/user: 1.4845434938892883


POPULARITY MODEL

In [4]:
popularity = (
    train_df.groupby("asin")
      .agg(
          avg_rating=("overall", "mean"),
          cnt=("overall", "count")
      )
      .reset_index()
)

popularity["score"] = popularity["avg_rating"] * popularity["cnt"]
popularity = popularity.sort_values("score", ascending=False)

pickle.dump(popularity, open("../model/popularity.pkl", "wb"))
print("✓ Popularity model saved")

✓ Popularity model saved


CONTENT-BASED 

In [5]:
# Create review text for TF-IDF (combine summary + reviewText, lowercase)
train_df["review_text"] = (
    train_df["summary"].fillna("") + ". " + train_df["reviewText"].fillna("")
).str.lower().str.strip()

print("Sample review text:")
print(train_df["review_text"].iloc[0][:200])

Sample review text:
a plastic surgeon that used it on his face lift patients recommend it. i had a real problem with my .... i have used jane iredale makeup for over 10 years. a plastic surgeon that used it on his face l


In [6]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    stop_words="english"
)

X = tfidf.fit_transform(train_df["review_text"])

In [7]:
svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X)

print("TF-IDF shape:", X.shape)
print("SVD shape:", X_svd.shape)

TF-IDF shape: (21076, 30000)
SVD shape: (21076, 100)


In [8]:
# Item representation = mean vector of all reviews
item_vectors = {}
for asin, vec in zip(train_df["asin"], X_svd):
    item_vectors.setdefault(asin, []).append(vec)

item_vectors = {
    k: np.mean(v, axis=0)
    for k, v in item_vectors.items()
}

pickle.dump(tfidf, open("../model/tfidf.pkl", "wb"))
pickle.dump(svd, open("../model/svd.pkl", "wb"))
pickle.dump(item_vectors, open("../model/content_vectors.pkl", "wb"))

print("✓ Content-based model saved")

✓ Content-based model saved


COLLABORATIVE FILTERING (Item-based)

https://files.grouplens.org/papers/www10_sarwar.pdf

In [9]:
train_df["interaction"] = (train_df["overall"] >= 4).astype(int)

ui = train_df.pivot_table(
    index="reviewerID",
    columns="asin",
    values="interaction",
    fill_value=0
)

item_sim = cosine_similarity(ui.T)
item_sim = pd.DataFrame(
    item_sim,
    index=ui.columns,
    columns=ui.columns
)

pickle.dump(ui, open("../model/ui_matrix.pkl", "wb"))
pickle.dump(item_sim, open("../model/itemknn.pkl", "wb"))

print("✓ ItemKNN model saved")

✓ ItemKNN model saved


HYBRID ENSEMBLE (300 candidates)

In [10]:
CB_W, CF_W, POP_W = 0.25, 0.65, 0.10
TOPK = 300

pop_score = dict(zip(popularity["asin"], popularity["score"]))

def hybrid_recommend(user_id):
    if user_id not in ui.index:
        return []

    seen_items = ui.loc[user_id]
    seen_items = seen_items[seen_items > 0].index.tolist()

    # CF
    cf_scores = item_sim[seen_items].mean(axis=1)

    # CB
    cb_scores = {}
    for i in seen_items:
        vi = item_vectors[i]
        for j, vj in item_vectors.items():
            cb_scores[j] = cb_scores.get(j, 0) + np.dot(vi, vj)

    # Normalize
    cf_scores = cf_scores / cf_scores.max()
    cb_scores = {k: v / max(cb_scores.values()) for k, v in cb_scores.items()}

    # Hybrid
    final_scores = {}
    for item in cf_scores.index:
        final_scores[item] = (
            CF_W * cf_scores.get(item, 0)
            + CB_W * cb_scores.get(item, 0)
            + POP_W * pop_score.get(item, 0) / popularity["score"].max()
        )

    return sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:TOPK]


GENERATE OUTPUT FOR ALL USERS

In [11]:
users = ui.index.tolist()
hybrid_output = {}

for u in users:
    hybrid_output[u] = hybrid_recommend(u)  # list[(item, score)] length=300

with open("../model/hybrid_candidates_baseline.pkl", "wb") as f:
    pickle.dump(hybrid_output, f)

print("✓ Hybrid output saved")
print("Users:", len(hybrid_output))


✓ Hybrid output saved
Users: 2782


EVALUATE RECALL@K

In [12]:
def recall_at_k(preds, ground_truth, k):
    """Macro-average recall: mean of per-user recall (matching ranking stage)."""
    user_recalls = []

    for u, gt_items in ground_truth.items():
        if u not in preds:
            continue

        pred_items = [i for i, _ in preds[u][:k]]
        user_recall = len(set(pred_items) & set(gt_items)) / len(gt_items)
        user_recalls.append(user_recall)

    return np.mean(user_recalls)

In [13]:
Ks = [10, 20, 30, 50, 100, 200, 300]

for k in Ks:
    r = recall_at_k(hybrid_output, ground_truth, k)
    print(f"Recall@{k}: {r:.4f}")


Recall@10: 0.2570
Recall@20: 0.3156
Recall@30: 0.3533
Recall@50: 0.3929
Recall@100: 0.4601
Recall@200: 0.5404
Recall@300: 0.6069
